# Customer Churn Prediction — Part 3: Model Training, Evaluation & Tuning
**Objective:** Train 4 classification algorithms, compare using precision/recall/F1/ROC-AUC, tune hyperparameters with GridSearchCV, and persist the production pipeline.

In [1]:
import sys
sys.path.append('..')
import joblib
import pandas as pd
from src.data_preprocessing import load_raw_data, prepare_train_test
from src.train import train_baseline_models, tune_best_model, save_pipeline
from src.predict import load_model, predict_single_customer, UNSEEN_CUSTOMERS

raw_df = load_raw_data('../data/dataset.csv')
X_train, X_test, y_train, y_test = prepare_train_test(raw_df, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

### Baseline Model Training
We evaluate 4 distinct algorithms on holdout test data:

In [2]:
baseline_results, baseline_pipelines = train_baseline_models(X_train, y_train, X_test, y_test)

comparison_df = pd.DataFrame(baseline_results).T[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
comparison_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
comparison_df.sort_values(by='ROC-AUC', ascending=False)

### Hyperparameter Tuning via GridSearchCV
Optimizing top performing model using 5-Fold Stratified Cross Validation.

In [3]:
best_pipeline, tuning_report = tune_best_model(X_train, y_train, X_test, y_test, model_name='Logistic Regression')
print(f"Best Hyperparameters: {tuning_report['best_params']}")
print(f"Best 5-Fold CV ROC-AUC: {tuning_report['best_cv_roc_auc']:.4f}")
print(f"Test ROC-AUC: {tuning_report['test_metrics']['roc_auc']:.4f}")

### Inference on Unseen Customers using Saved Joblib Pipeline

In [4]:
save_pipeline(best_pipeline, '../models/customer_churn_model.joblib')
loaded_pipe = load_model('../models/customer_churn_model.joblib')

for cust in UNSEEN_CUSTOMERS:
    res = predict_single_customer(cust, model=loaded_pipe)
    print(f"[{cust['id']}] {cust['profile_name']}: {res['prediction']} ({res['churn_probability_percentage']})")
    print(f"   Drivers: {res['diagnostics']['key_risk_drivers']}")